©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、PyTorchのnn.RNNおよびnn.RNNを用いて、通常のRNNと双方向RNN（Bidirectional RNN）を実装します。MNISTの画像を行単位のシーケンスとして入力し、RNNで分類するタスクを通じて、双方向処理による精度向上の効果を比較・確認します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
%%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

!pip uninstall torch -y
!pip install torch==2.7.0

!pip uninstall torchvision -y
!pip install torchvision==0.22.0

## RNNと双方向RNNの実装

In [ ]:
# Imports
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import (
    DataLoader,
)
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ハイパーパラメータの定義
input_size = 28           # 入力特徴のサイズ（例えば、MNISTの場合、画像の各行は28ピクセル）
sequence_length = 28      # シーケンスの長さ（例えば、MNISTの場合、28行のシーケンスがある）
num_layers = 2            # RNN層の数
hidden_size = 128         # 各RNN層の隠れユニットの数
num_classes = 10          # 分類するクラスの数（例えば、MNISTの場合、0から9までの10クラス）
learning_rate = 0.001     # 学習率（モデルのパラメータ更新の速度を制御）
batch_size = 64           # 一度に学習するデータの数（バッチサイズ）
num_epochs = 100            # 訓練データ全体を何回繰り返して学習するか（エポック数）

## データのロード

In [ ]:
train_dataset = datasets.MNIST(
    root="dataset/", train=True, transform=transforms.ToTensor(), download=True
)
# 訓練用データセットとしてMNISTを読み込む

test_dataset = datasets.MNIST(
    root="dataset/", train=False, transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
# 学習時に使うDataLoader

test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)
# 評価用のDataLoader

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 520kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.1MB/s]


## RNN

In [ ]:
class RNN(nn.Module):
    # シンプルなRNNでMNISTを分類するモデル
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size          # 隠れ状態の次元数を保持
        self.num_layers = num_layers            # RNNの層数を保持
        self.sequence_length = sequence_length  # 何タイムステップ処理するかを保持
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size * sequence_length, num_classes)

    def forward(self, x):
        # 初期化
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)

        # 順伝播
        out, _ = self.rnn(x, h0)  # RNN には x と h0 を渡す

        out = out.reshape(out.shape[0], -1)

        # 最後の時間ステップの隠れ層をデコードする
        out = self.fc(out)  # 変更なし
        return out

In [ ]:
# networkの初期化
model = RNN(input_size, hidden_size, num_layers, num_classes, sequence_length).to(device)

In [ ]:
# 損失関数と最適化関数
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Networkの訓練
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        # データをcudaに取得する
        data = data.to(device=device).squeeze(1)
        targets = targets.to(device=device)

        # 順伝播
        scores = model(data)
        loss = criterion(scores, targets)

        # 逆伝播
        optimizer.zero_grad()
        loss.backward()

        # 勾配計算
        optimizer.step()

In [ ]:
# 学習とテストで精度を確認
def check_accuracy(loader, model):
    # データローダーが学習データセットを使用しているかテストデータセットを使用しているかを判定し、適切なメッセージを表示
    if loader.dataset.train:
        print("Checking accuracy on training data")
    else:
        print("Checking accuracy on test data")

    num_correct = 0       # 正しく予測されたサンプル数
    num_samples = 0       # 総サンプル数
    model.eval()          # モデルを評価モードに設定

    with torch.no_grad():     # 勾配計算を行わない
        for x, y in loader:   # データローダーからバッチごとにデータを取得
            x = x.to(device=device).squeeze(1)    # データをデバイスに移動し、不要な次元を削除
            y = y.to(device=device)               # ラベルをデバイスに移動

            scores = model(x)                 # モデルを使用して予測スコアを計算
            _, predictions = scores.max(1)    # 最大スコアを持つクラスを予測として選択
            num_correct += (predictions == y).sum()   # 予測が正しいかどうかをチェックし、正しい数をカウント
            num_samples += predictions.size(0)        # 処理したサンプル数をカウント

        # 精度を計算し、表示
        print(
            f"Got {num_correct} / {num_samples} with accuracy  \
              {float(num_correct)/float(num_samples)*100:.2f}"
        )

    model.train()   # モデルを学習モードに戻す

# 学習データローダーを使って学習データでの精度をチェック
check_accuracy(train_loader, model)

# テストデータローダーを使ってテストデータでの精度をチェック
check_accuracy(test_loader, model)

Checking accuracy on training data
Got 59576 / 60000 with accuracy                99.29
Checking accuracy on test data
Got 9839 / 10000 with accuracy                98.39


## 双方向RNN

In [ ]:
# ハイパーパラメータの定義
input_size = 28           # 入力特徴のサイズ（例えば、MNISTの場合、画像の各行は28ピクセル）
sequence_length = 28      # シーケンスの長さ（例えば、MNISTの場合、28行のシーケンスがある）
num_layers = 1            # RNN層の数
hidden_size = 128         # 各RNN層の隠れユニットの数
num_classes = 10          # 分類するクラスの数（例えば、MNISTの場合、0から9までの10クラス）
learning_rate = 0.01     # 学習率（モデルのパラメータ更新の速度を制御）
batch_size = 128           # 一度に学習するデータの数（バッチサイズ）
num_epochs = 50            # 訓練データ全体を何回繰り返して学習するか（エポック数）

In [ ]:
class BRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # 双方向RNNの初期化
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        # 隠れ層のサイズが2倍になるので、Linear層の入力サイズもそれに合わせて調整
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # h0の初期化
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(device)  # 双方向なので2倍

        # RNN層にxとh0を渡す
        out, _ = self.rnn(x, h0)

        # 双方向RNNの全時刻の出力を活用
        out = self.fc(out.mean(dim=1))

        return out

In [ ]:
# networkの初期化
model = BRNN(input_size, hidden_size, num_layers, num_classes).to(device)

In [ ]:
# 勾配クリッピングの閾値を設定
clip_value = 5.0

for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        # データをデバイスに転送する際の次元削減が必要か確認する
        data = data.to(device=device)  # 不必要な次元削減を削除
        targets = targets.to(device=device)

        # 順伝播
        data = data.view(data.size(0), data.size(2), -1)
        scores = model(data)
        loss = criterion(scores, targets)

        # 逆伝播
        optimizer.zero_grad()
        loss.backward()

        # 勾配クリッピングを適用
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)

        # 勾配計算
        optimizer.step()

In [ ]:
# 学習とテストで精度を確認
def check_accuracy(loader, model):

    # ローダーからのデータセットが学習用かテスト用かを判断し、対応するメッセージを表示
    if loader.dataset.train:
        print("Checking accuracy on training data")
    else:
        print("Checking accuracy on test data")

    num_correct = 0       # 正しく予測されたデータの数
    num_samples = 0       # 総データ数
    model.eval()          # モデルを評価モードに設定

    with torch.no_grad():         # 勾配計算を行わない
        for x, y in loader:       # データローダーからバッチごとにデータを取得
            x = x.to(device=device).squeeze(1)      # データをデバイスに移動し、不要な次元を削除
            y = y.to(device=device)                 # ラベルをデバイスに移動

            scores = model(x)                       # モデルを使用して予測スコアを計算
            _, predictions = scores.max(1)          # 最大スコアを持つクラスを予測として選択
            num_correct += (predictions == y).sum() # 予測が正しいかどうかをチェックし、正しい数をカウント
            num_samples += predictions.size(0)      # 処理したサンプル数をカウント

        # 精度を計算し、表示
        print(
            f"Got {num_correct} / {num_samples} with accuracy  \
              {float(num_correct)/float(num_samples)*100:.2f}"
        )

    model.train()

# 学習データローダーを使って学習データでの精度をチェック
check_accuracy(train_loader, model)

# テストデータローダーを使ってテストデータでの精度をチェック
check_accuracy(test_loader, model)

Checking accuracy on training data
Got 6600 / 60000 with accuracy                11.00
Checking accuracy on test data
Got 1116 / 10000 with accuracy                11.16


## 🔧 実践問題1：nn.RNNをnn.GRUに置き換えて精度を比較する

上の通常RNNモデルでは `nn.RNN` を使用しています。
しかし、通常のRNNは長い系列で**勾配消失**が起きやすく、時系列の長距離依存関係を学習しにくいという問題があります。

GRU（Gated Recurrent Unit）はゲート機構により勾配の流れを制御し、この問題を軽減します。

| モジュール | 特徴 | 隠れ状態 |
|:---|:---|:---|
| `nn.RNN` | シンプルだが勾配消失しやすい | `h` のみ |
| `nn.GRU` | ゲート機構で勾配消失を軽減。RNNと同じAPIで使える | `h` のみ |
| `nn.LSTM` | 忘却ゲート+セル状態で長期記憶が可能。戻り値が異なる | `(h, c)` の2つ |

---

**問題：** 以下のコードの `______` を埋めて、`nn.RNN` を `nn.GRU` に置き換えたモデルを作成し、精度を比較してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
GRUはRNNと同じく隠れ状態 <code>h</code> だけを使うため、<code>forward</code> の書き方はほぼ同じです。変更が必要なのはどこか考えてみてください。
</blockquote>

</details>

<br/>


In [ ]:
class RNN_GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length):
        super(RNN_GRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.sequence_length = sequence_length
        self.rnn = nn.______(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size * sequence_length, num_classes)

    def forward(self, x):
        # GRUの隠れ状態の初期化（RNNと同じ形状）
        h0 = torch.zeros(self.______, x.size(0), self.______).to(device)
        # GRUの戻り値はRNNと同じ: (output, h_n)
        out, _ = self.rnn(x, ______)
        out = out.reshape(out.shape[0], -1)
        out = self.fc(out)
        return out


# モデル作成・学習
model_gru = RNN_GRU(input_size, hidden_size, num_layers, num_classes, sequence_length).to(device)
optimizer_gru = optim.Adam(model_gru.parameters(), lr=0.001)

for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device).squeeze(1)
        targets = targets.to(device=device)
        scores = model_gru(data)
        loss = criterion(scores, targets)
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()

print('--- GRU Model ---')
check_accuracy(train_loader, model_gru)
check_accuracy(test_loader, model_gru)


<details><summary>解答例</summary>

```python
self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
out, _ = self.rnn(x, h0)
```

- `nn.GRU` は `nn.RNN` と**完全に同じAPI**で使えます。コンストラクタの引数も戻り値 `(output, h_n)` も同じ形式です
- `h0` の形状 `(num_layers, batch_size, hidden_size)` もRNNと同一です
- 違いは内部のゲート機構だけ：GRUはリセットゲートと更新ゲートにより「過去の情報をどれだけ保持するか」を学習的に制御します
- 一般にGRUはRNNより高い精度が出ます。MNISTは系列長28と短めですが、それでも差が出ることがあります
- `nn.LSTM` に変える場合は戻り値が `(output, (h_n, c_n))` のタプルになるため、`h0` に加えて `c0` も必要になります。これが実践問題2のテーマです
</details>


## 🔧 実践問題2：双方向RNNをnn.LSTMに置き換える

上のBRNNモデルでは `nn.RNN(bidirectional=True)` を使用しています。
これを `nn.LSTM` に置き換えて、双方向LSTMを実装します。

LSTMはRNN/GRUと異なり、隠れ状態 `h` に加えて**セル状態 `c`** を持ちます：

| | nn.RNN / nn.GRU | nn.LSTM |
|:---|:---|:---|
| 初期状態 | `h0` のみ | `h0` と `c0` の**両方**が必要 |
| 戻り値 | `(output, h_n)` | `(output, (h_n, c_n))` |
| 双方向時のh0の形状 | `(num_layers * 2, batch, hidden)` | 同じ（c0も同じ形状） |

---

**問題：** 以下のコードの `______` を埋めて、双方向LSTMモデルを実装してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
<code>c0</code>（セル状態の初期値）は <code>h0</code> と同じ形状で0初期化します。LSTMには <code>(h0, c0)</code> をタプルで渡します。
</blockquote>

</details>

<br/>

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # TODO: 双方向LSTMを定義
        self.lstm = nn.______(input_size, hidden_size, num_layers,
                              batch_first=True, bidirectional=______)
        # 双方向なので出力の次元数は hidden_size の何倍？
        self.fc = nn.Linear(hidden_size * ______, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * ______, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers * ______, x.size(0), self.hidden_size).to(device)

        # LSTMには (h0, c0) をタプルで渡す
        out, (______, ______) = self.lstm(x, (h0, c0))

        # 全時刻の出力の平均をとってから分類
        out = self.fc(out.mean(dim=1))
        return out


# モデル作成・学習
model_bilstm = BiLSTM(input_size, hidden_size, 1, num_classes).to(device)
optimizer_bl = optim.Adam(model_bilstm.parameters(), lr=0.01)

for epoch in range(50):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device)
        targets = targets.to(device=device)
        data = data.view(data.size(0), data.size(2), -1)
        scores = model_bilstm(data)
        loss = criterion(scores, targets)
        optimizer_bl.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bilstm.parameters(), 5.0)
        optimizer_bl.step()

print('--- Bidirectional LSTM ---')
check_accuracy(train_loader, model_bilstm)
check_accuracy(test_loader, model_bilstm)


<details><summary>解答例</summary>

```python
self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                    batch_first=True, bidirectional=True)
self.fc = nn.Linear(hidden_size * 2, num_classes)

h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(device)
c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(device)

out, (h_n, c_n) = self.lstm(x, (h0, c0))
```

- `nn.LSTM` のコンストラクタは `nn.RNN` と同じ引数 + `bidirectional=True` で双方向になります
- `hidden_size * 2` は双方向（forward方向 + backward方向）の出力が結合されるためです
- `h0`, `c0` ともに形状 `(num_layers * 2, batch, hidden_size)` です。`* 2` は双方向の分です
- LSTMの戻り値は `(output, (h_n, c_n))` のタプルです。RNN/GRUの `(output, h_n)` と異なり、**セル状態 `c_n` も返ってくる**点が最大の違いです
- 双方向LSTM + 勾配クリッピングの組み合わせは、自然言語処理の多くのタスクで標準的なベースラインとして使われています
</details>
